In [3]:
import torch
import os
import soundfile as sf
from encodec import EncodecModel
from encodec.utils import convert_audio

print("Loading EnCodec...")
model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(6.0)

wav_np, sr = sf.read("./sample_data/sample.wav")
wav = torch.from_numpy(wav_np).float().unsqueeze(0)
if wav.ndim == 2: wav = wav.unsqueeze(0)
wav = convert_audio(wav, sr, 24000, 1)


with torch.no_grad():
    encoded_frames = model.encode(wav)

codes = torch.cat([encoded[0] for encoded in encoded_frames], dim=-1)
codes_layer0 = codes[:, :1, :]
codes_layer01 = codes[:, :2, :]
codes_layer02 = codes[:, :3, :]


print(f"All layers: {codes.shape}")
print(f"Layer 0 only:  {codes_layer0.shape}")

with torch.no_grad():
    decoded_full = model.decode([(codes, None)])
    decoded_layer0 = model.decode([(codes_layer0, None)])
    decoded_layer01 = model.decode([(codes_layer01, None)])
    decoded_layer02 = model.decode([(codes_layer02, None)])

original_np = decoded_full.squeeze().cpu().numpy()
layer0_np = decoded_layer0.squeeze().cpu().numpy()
layer01_np = decoded_layer01.squeeze().cpu().numpy()
layer02_np = decoded_layer02.squeeze().cpu().numpy()


os.makedirs("reconstructed", exist_ok=True)

sf.write("reconstructed/sample_reconstructed_full.wav", original_np, 24000)
sf.write("reconstructed/sample_reconstructed_layer0.wav", layer0_np, 24000)
sf.write("reconstructed/sample_reconstructed_layer01.wav", layer01_np, 24000)
sf.write("reconstructed/sample_reconstructed_layer02.wav", layer02_np, 24000)


Loading EnCodec...


c:\Users\alima\Desktop\janek\tts\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


All layers: torch.Size([1, 8, 630])
Layer 0 only:  torch.Size([1, 1, 630])
